# 4 · Semantic Content Blocker: filter a feed by meaning

## The problem

Ad blockers and content filters run on brittle signals: domain lists, CSS selectors, keyword regexes. Reword the ad
("this one weird app..."), drop the word *sponsored*, or post the spoiler as a plain sentence, and the filter misses it.
And every user wants something different: one person hates crypto spam, another just wants to avoid spoilers.

## Why Jev

Jev is a **System One** model: it returns **typed judgments with calibrated probabilities**, in about 100 ms, for a
fraction of a cent. A feed filter runs on **every** item a user scrolls past, so it must be fast and nearly free, and it
must return **numbers you can threshold**, not a paragraph. That's exactly Jev's shape.

We judge each item by what it **means**: one yes/no question per category the user switched on. The user's settings
*are* the list of questions, so changing the policy means flipping a toggle, not retraining anything.

## What we'll build

```
feed item ──► Jev: one Noul per ENABLED category       ──► policy in code              ──► HIDE / BLUR / DIM / SHOW
               "is it an ad?"           0.98                • worst category > 0.80 → HIDE
               "is it crypto shilling?" 0.05                  (spoiler → BLUR instead)
               "is it a spoiler?"       0.02                • worst > 0.50 → DIM
               + intrusiveness (Score 0..2)                 • otherwise SHOW
```

## What you'll learn
- Building the question set **from user settings**: the toggles decide which `Noul`s get asked.
- Reading a batch of `Noul`s plus a `Score` from one response.
- Writing a small, readable **policy** (take the worst category, threshold it, special-case spoilers) in plain Python.

> **How to run this notebook**
> 1. Pick the **`.venv`** kernel (top right in VS Code, *Kernel → Change kernel* in Jupyter).
> 2. Run the cells **top to bottom** (or *Run All*). Every cell builds on the ones above it.
> 3. If you see `NameError: name '...' is not defined`, the kernel was restarted (switching kernels does this) and forgot the earlier cells. Click **Run All** or *Run All Above*, then continue.

## Step 0 · Setup

This cell:
- loads the API key from the `.env` file at the repo root (`TYPESAFE_API_KEY`),
- creates the Jev client `jev`. Every judgment in this notebook goes through it,
- imports the Jev question types (`Noul` and `Score` are used here),
- defines `bar()`, a tiny helper that draws a 0..1 probability as a text bar so numbers are easy to eyeball.

In [1]:
import os
from pathlib import Path
from dotenv import load_dotenv
from typesafe_sdk import TypeSafeClient, Noul, NoulCriteria, Score, Choice

# Look for a .env file in this folder or any parent folder (TYPESAFE_API_KEY)
for folder in [Path.cwd(), *Path.cwd().parents]:
    if (folder / ".env").exists():
        load_dotenv(folder / ".env")
        break

jev = TypeSafeClient()   # the judge: fast, cheap, typed answers


def bar(p, width=24):
    """A unicode progress bar for a 0..1 value."""
    filled = round(p * width)
    return "█" * filled + "·" * (width - filled)


print("client ready")

client ready


## Step 1 · The categories, the user's toggles, and the thresholds

`CATEGORIES` is everything the blocker *can* detect, each with a one-line definition (that definition is what Jev
judges against). `ENABLED` is what this user has switched **on**. The thresholds turn probabilities into actions:

| Worst category probability | Action | Why |
|---|---|---|
| above `HIDE` (0.80), category is `spoiler` | **BLUR** | spoilers stay clickable: the user may want to reveal them later |
| above `HIDE` (0.80), any other category | **HIDE** | confidently unwanted |
| above `DIM` (0.50) | **DIM** | probably unwanted, but not sure enough to remove |
| 0.50 or below | **SHOW** | |

In [2]:
CATEGORIES = {
    "ad": "A paid advertisement, sponsored post, or promotion",
    "ragebait": "Content engineered to provoke outrage or anger for engagement",
    "crypto_shill": "Promotion of crypto tokens, NFTs, or get-rich schemes",
    "spoiler": "Reveals plot or results of a show, game, or sporting event",
    "ai_slop": "Low-effort AI-generated filler with no real substance",
}

# What the user has switched ON. Step 6 shows what happens when one is switched off.
ENABLED = ["ad", "ragebait", "crypto_shill", "spoiler", "ai_slop"]

HIDE = 0.80   # worst category above this -> HIDE (or BLUR for spoilers)
DIM = 0.50    # worst category above this -> DIM
print("enabled:", ENABLED)

enabled: ['ad', 'ragebait', 'crypto_shill', 'spoiler', 'ai_slop']


## Step 2 · Build the questions from the toggles

**New Jev concept: `Noul`.** A `Noul` is a yes/no question; its answer `.noul` is the **probability of yes**, 0..1.
Calibrated means you can put a threshold on it: 0.9 means "yes about 9 times in 10".

**New Jev concept: `Score`.** A `Score` has an **ordered rubric** in `criteria`: level 0 is the first entry, level 2
the last. `.score` is a *continuous* position on that scale (e.g. 1.6 = between "mildly distracting" and "pure noise"),
plus a `.confidence`.

`build_questions()` makes **one `Noul` per enabled category** plus one `Score` for overall intrusiveness. It's a function,
not a constant, because the questions depend on `ENABLED`: switch a category off and its question simply isn't asked.

This cell only **defines** the function. Nothing is sent to Jev yet.

In [3]:
def build_questions():
    q = {c: Noul(instructions=f"Does this item fit the category '{c}': {CATEGORIES[c]}?") for c in ENABLED}
    q["intrusiveness"] = Score(
        instructions="How intrusive or low-value is this item to a focused reader?",
        criteria=["Genuinely useful", "Mildly distracting", "Pure noise or manipulation"],   # 0, 1, 2
    )
    return q


print("questions that would be asked right now:", list(build_questions()))

questions that would be asked right now: ['ad', 'ragebait', 'crypto_shill', 'spoiler', 'ai_slop', 'intrusiveness']


## Step 3 · Ask once and read the raw answer

The `state` is the feed item's text. One `system_one` call answers all six questions. `Noul` answers are in `r.nouls[...]`,
the `Score` is in `r.scores[...]`.

In [4]:
item = "$PEPE2 is about to 100x. Last chance before launch. Not financial advice but WAGMI."
r = jev.system_one(state=item, questions=build_questions())

print(item, "\n")
for c in ENABLED:
    print(f"  {c:<13} {bar(r.nouls[c].noul, 20)} {r.nouls[c].noul:.2f}")
s = r.scores["intrusiveness"]
print(f"  intrusiveness {s.score:.2f} / 2   (confidence {s.confidence:.2f})")

$PEPE2 is about to 100x. Last chance before launch. Not financial advice but WAGMI. 

  ad            █████████████████··· 0.84
  ragebait      ████················ 0.20
  crypto_shill  ████████████████████ 0.98
  spoiler       ···················· 0.02
  ai_slop       █████████████······· 0.66
  intrusiveness 2.00 / 2   (confidence 1.00)


**What to notice**
- `crypto_shill` is near 1, as you'd expect.
- Other categories can be high **at the same time** (this post also reads like an ad). The questions are independent, so the probabilities don't add up to 1.
- `intrusiveness` sits at or near the top of its 0..2 scale: "pure noise or manipulation".
- Spoiler and ragebait stay low: the model isn't just saying "bad post" to everything.

## Step 4 · The policy, in plain Python

`judge()` asks the questions, finds the **worst** (highest-probability) category, and applies the threshold table from
Step 1. Every decision lives here, in code you can read and test. The guard at the top gives a clear message if the
kernel was restarted.

`intrusiveness` is returned too, but the policy doesn't use it yet. That's one of the exercises.

In [5]:
def judge(item, client):
    if "CATEGORIES" not in globals():
        raise RuntimeError("CATEGORIES is missing: the kernel restarted. Use 'Run All' (or run the cells above) first.")

    r = client.system_one(state=item, questions=build_questions())
    scores = {c: r.nouls[c].noul for c in ENABLED}
    worst_cat = max(scores, key=scores.get)
    worst = scores[worst_cat]

    if worst > HIDE:
        action = "BLUR" if worst_cat == "spoiler" else "HIDE"
    elif worst > DIM:
        action = "DIM"
    else:
        action = "SHOW"
    return {"action": action, "cat": worst_cat, "p": worst, "scores": scores,
            "intrusiveness": r.scores["intrusiveness"].score}


print("judge() defined. HIDE =", HIDE, " DIM =", DIM)

judge() defined. HIDE = 0.8  DIM = 0.5


## Step 5 · Run it on a feed

Eight items: an ad, ragebait, a crypto shill, a TV spoiler, some AI filler, a borderline clickbait headline, and two
normal, useful posts. One Jev request per item. For each we print the action and the worst category.

In [6]:
FEED = [
    "Breaking: scientists confirm 8 glasses of water a day myth finally debunked in new study.",
    "SPONSORED - This one weird productivity app replaced my entire team. Sign up free today!",
    "They DON'T want you to know this. The elites are HIDING the truth about your groceries.",
    "$PEPE2 is about to 100x. Last chance before launch. Not financial advice but WAGMI.",
    "In tonight's finale, the detective turns out to be the killer all along. Wild ending.",
    "Here are 10 tips to improve your morning routine. Tip 1: wake up. Tip 2: be productive...",
    "Our team just shipped incremental static regeneration for the docs site. Here's how it works.",
    "Reminder: the library closes at 6pm on Fridays during the summer.",
]
verdicts = [judge(it, jev) for it in FEED]
for it, d in zip(FEED, verdicts):
    print(f"[{d['action']:^5}] {it[:62]:<62}  {d['cat']} {d['p']:.2f}")

[ DIM ] Breaking: scientists confirm 8 glasses of water a day myth fin  ai_slop 0.76
[HIDE ] SPONSORED - This one weird productivity app replaced my entire  ad 0.98
[HIDE ] They DON'T want you to know this. The elites are HIDING the tr  ragebait 0.93
[HIDE ] $PEPE2 is about to 100x. Last chance before launch. Not financ  crypto_shill 0.98
[BLUR ] In tonight's finale, the detective turns out to be the killer   spoiler 0.98
[HIDE ] Here are 10 tips to improve your morning routine. Tip 1: wake   ai_slop 0.92
[SHOW ] Our team just shipped incremental static regeneration for the   ai_slop 0.24
[SHOW ] Reminder: the library closes at 6pm on Fridays during the summ  ai_slop 0.22


**What to notice**
- The ad, the ragebait, the crypto shill and the "10 tips" filler are **hidden**, each for a different category, all decided by what the text means rather than by a keyword list.
- The TV finale is **blurred**, not hidden: same threshold, but the policy special-cases spoilers.
- The clickbait-style "Breaking:" headline sits in the middle band, so it's **dimmed** rather than hidden. That's the job of the DIM band: "probably low value, not sure enough to remove".
- The two useful posts are **shown**: their worst category stays well below 0.5.

## Step 6 · Flip a toggle: the policy *is* the question list

Suppose this user doesn't mind spoilers. We switch `spoiler` off, judge the TV-finale item again, then switch it back on.
No model changes, no retraining: the spoiler question simply isn't asked any more.

In [7]:
finale = FEED[4]
saved = ENABLED
ENABLED = [c for c in saved if c != "spoiler"]          # the user flips the spoiler toggle off
print("enabled:", ENABLED)
print("questions asked:", list(build_questions()))
d = judge(finale, jev)
print(f"\n[{d['action']:^5}] {finale}\n        worst remaining category: {d['cat']} {d['p']:.2f}")

ENABLED = saved                                          # switch it back on for the rest of the notebook
print("\nrestored:", ENABLED)

enabled: ['ad', 'ragebait', 'crypto_shill', 'ai_slop']
questions asked: ['ad', 'ragebait', 'crypto_shill', 'ai_slop', 'intrusiveness']

[ DIM ] In tonight's finale, the detective turns out to be the killer all along. Wild ending.
        worst remaining category: ai_slop 0.54

restored: ['ad', 'ragebait', 'crypto_shill', 'spoiler', 'ai_slop']


**What to notice**
- With `spoiler` off, only four `Noul`s are sent (plus intrusiveness), and the finale item is **no longer blurred**.
- It is now judged only by the remaining categories. Its worst one is `ai_slop`, close to the 0.5 DIM line, so depending on the run it comes out DIM or SHOW. A short, punchy post looks a little like filler, and the DIM band handles that uncertainty by fading rather than hiding.
- This is the whole "user-configurable policy" idea: settings choose which questions get asked.

## Step 7 · Give it a face

Everything from here on is **presentation only**: a small style kit, a text box wired up with `ipywidgets`, and an HTML
renderer for feed items. `judge()` is unchanged. The feed below reuses the verdicts from Step 5, so it makes no extra API calls.

In [8]:
import ipywidgets as widgets
from IPython.display import HTML, display

# small style kit
INK, SLATE, CORAL, SAGE, AMBER, RED, LINE = "#1F1E1D", "#5E6B7A", "#D97757", "#1F7A3D", "#C98A00", "#C0392B", "#DEDACE"
CARD = f"background:#FAF9F5;border:1px solid {LINE};border-radius:10px;padding:14px 16px;margin:8px 0;"
BADGE = "display:inline-block;padding:2px 10px;border-radius:12px;color:white;font-weight:600;font-size:12px;"


def box(inner):
    return f'<div style="font-family:-apple-system,Segoe UI,sans-serif;color:{INK};max-width:820px;">{inner}</div>'


def text_app(placeholder, render, button="Run", value=""):
    """A text box + button. On submit, call render(text) and show the HTML it returns."""
    inp = widgets.Text(value=value, placeholder=placeholder, continuous_update=False,
                       layout=widgets.Layout(width="68%"))
    btn = widgets.Button(description=button, button_style="warning")
    out = widgets.Output()

    def go(_):
        with out:
            out.clear_output()
            if inp.value.strip():
                display(HTML(box("<i>thinking...</i>")))
                html = render(inp.value.strip())
                out.clear_output(wait=True)
                display(HTML(html))

    btn.on_click(go)
    inp.observe(go, names="value")   # fires when you press Enter
    display(widgets.HBox([inp, btn]), out)

`feed_item_html()` turns one verdict into a feed card: **HIDE** collapses it to a grey placeholder, **BLUR** blurs the
text (a real app would reveal it on click), **DIM** fades it, **SHOW** leaves it alone. Then we draw the whole feed.

In [9]:
def feed_item_html(item, d):
    a = d["action"]
    if a == "HIDE":
        return f'<div style="{CARD}opacity:.6;background:#efece4;color:{SLATE};font-style:italic;">[hidden: {d["cat"]} {d["p"]:.2f}]</div>'
    if a == "BLUR":
        return (f'<div style="{CARD}"><span style="filter:blur(4px);">{item}</span>'
                f'<div style="color:{SLATE};font-size:12px;">blurred spoiler {d["p"]:.2f}</div></div>')
    if a == "DIM":
        return f'<div style="{CARD}opacity:.45;">{item}<span style="color:{SLATE};font-size:12px;"> (dimmed {d["cat"]} {d["p"]:.2f})</span></div>'
    return f'<div style="{CARD}">{item}</div>'


# reuse the verdicts from Step 5, no extra API calls
display(HTML(box(f'<div style="color:{SLATE};font-size:13px;">blocking: {", ".join(ENABLED)}</div>'
                 + "".join(feed_item_html(it, d) for it, d in zip(FEED, verdicts)))))

## Step 8 · Try it yourself

Paste a post into the box and press **Enter** or click the button (needs a live kernel). Good ones to try in a workshop:
- `BUY $DOGE now, 100x guaranteed!` → **HIDE** as crypto_shill.
- `Use code SAVE20 for 20% off your first order` → **HIDE** as an ad, with no "sponsored" label anywhere.
- `Spoiler-free review: the new season is worth watching` → should **not** be blurred: it mentions a show but reveals nothing. It may still get DIMmed, because a one-line review blurb looks a little like filler. A good item to discuss the DIM band.
- `The final score was 3-1, Madrid won it in extra time` → **BLUR** as a sports spoiler.

In [10]:
text_app("paste a post, e.g. 'BUY $DOGE now, 100x guaranteed!'", lambda t: box(feed_item_html(t, judge(t, jev))), "Screen")

Output()

## Recap and exercises

**Key ideas**
1. Filter by **meaning**, not keywords: one `Noul` per category catches rewordings a regex never would.
2. **User settings = the question list.** Switching a category off removes its question; nothing is retrained.
3. **The model measures, the code decides.** Jev returned probabilities; "take the worst", the thresholds, and "spoilers blur" are plain Python.
4. A middle band (DIM) is a cheap way to handle uncertainty without hiding things the user might want.

**Exercises**
- Lower `HIDE` to 0.6 and re-run Step 5. Which items move from DIM to HIDE?
- Add a category: `"politics": "Partisan political commentary or campaigning"`, enable it, and write a post to test it.
- Use `intrusiveness`: DIM any item with `intrusiveness > 1.5` even if no category fires.
- Give spoilers their own threshold (e.g. `SPOILER_BLUR = 0.6`), since a missed spoiler is worse than a missed ad.